# Generative Backend — Unified Action Pipeline

Demonstrates the universal `ActionPipeline` handling **both PickUp and Place**
from a single entry point without knowing the action type in advance.

**Key design points:**
- One pipeline class, any action type
- One slot-filling LLM call classifies the action **and** extracts parameters
- The result is a clean per-action schema: `PickUpSlotSchema` or `PlaceSlotSchema`
- An `ActionDispatcher` registry routes each schema to the correct handler
- A **generic resolver node** (`workflows/nodes/resolver.py`) handles Phase 2 for all
  action types — new actions only need a prompt and a schema class

**Package layout (relevant):**
```
llmr/
├── action_pipeline.py          ← entry point
├── action_dispatcher.py        ← handler registry
├── entity_grounder.py          ← NL description → world Body
└── workflows/
    ├── nodes/
    │   ├── slot_filler.py      ← Phase 1: classify + extract (all actions)
    │   └── resolver.py         ← Phase 2: discrete resolution (generic, all actions)
    ├── schemas/
    │   ├── common.py           ← EntityDescriptionSchema (shared across all actions)
    │   ├── pick_up.py
    │   └── place.py
    └── prompts/
        ├── pick_up.py
        └── place.py
```

**Stages shown:**
1. **Slot filling** — one LLM call returns a typed per-action schema
2. **PickUp full run** — end-to-end `PickUpAction` via `pipeline.run()`
3. **Place full run** — end-to-end `PlaceAction` via the same `pipeline.run()`
4. **Step-by-step breakdown** — `classify_and_extract()` → `dispatch()` for both
5. **Generic resolver** — calling `run_resolver()` directly with any prompt + schema

**Prerequisites:** `cd cognitive_robot_abstract_machine && uv sync --active`  
**API key:** set `OPENAI_API_KEY` in `llmr/.env`

## 1 · Environment & Imports

In [ ]:
import logging
import pathlib
from dotenv import load_dotenv

from pycram.datastructures.grasp import GraspDescription

# Load API keys before any LLM import
_env = pathlib.Path().resolve().parent / ".env"
load_dotenv(_env, override=True)

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")

# ── SDT / pycram stack ────────────────────────────────────────────────────────
from semantic_digital_twin.robots.pr2 import PR2
from pycram.datastructures.dataclasses import Context
from pycram.testing import setup_world
from pycram.language import SequentialPlan
from pycram.motion_executor import simulated_robot

# ── Generative backend — pipeline entry point ─────────────────────────────────
from llmr.pipeline import ActionPipeline

# ── Schemas (common + action-specific) ───────────────────────────────────────
from llmr.workflows.schemas.common import EntityDescriptionSchema
from llmr.workflows.schemas.pick_up import PickUpSlotSchema
from llmr.workflows.schemas.place import PlaceSlotSchema

# ── LangGraph nodes (Phase 1 + generic Phase 2) ───────────────────────────────
from llmr.workflows.nodes.slot_filler import run_slot_filler
from llmr.workflows.nodes.resolver import (
    run_resolver,
    run_pickup_resolver,
    run_place_resolver,
)
logging.getLogger("llmr").setLevel(logging.DEBUG)
print("All imports OK")

## 2 · Load World

`setup_world()` loads the apartment URDF, a PR2 robot, a milk carton (`milk.stl`),
and a breakfast cereal box. The milk sits on the kitchen counter.

We will use:
- **Pick-up instruction** — grab the milk from the counter
- **Place instruction** — put the milk on the table

In [ ]:
world = setup_world()

try:
    robot = PR2.from_world(world)
except Exception as exc:
    print(f"Note: PR2 full setup skipped ({type(exc).__name__}) — recovering annotation")
    robot = next((a for a in world.semantic_annotations if isinstance(a, PR2)), None)
    if robot is None:
        raise RuntimeError("Could not obtain PR2 annotation") from exc
    with world.modify_world():
        robot._setup_velocity_limits()
        robot._setup_hardware_interfaces()
        robot._setup_joint_states()

context = Context(world, robot)
# Confirm key bodies are present
milk_body = world.get_body_by_name("milk.stl")
milk_pos  = milk_body.global_pose.to_position()
print(f"World loaded")
print(f"  milk  → x={float(milk_pos.x):.2f}, y={float(milk_pos.y):.2f}, z={float(milk_pos.z):.2f}")

# Show all bodies so we can pick an appropriate place target
all_body_names = [
    str(getattr(getattr(b, "name", None), "name", b))
    for b in world.bodies
]
print(f"  bodies present ({len(all_body_names)}): {', '.join(all_body_names[:20])}")

In [ ]:
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
import threading
import rclpy

rclpy.init()
_ros_node = rclpy.create_node('semantic_digital_twin')
_ros_thread = threading.Thread(target=rclpy.spin, args=(_ros_node,), daemon=True)
_ros_thread.start()

In [ ]:
_tf_publisher  = TFPublisher(_world=world, node=_ros_node)
_viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)
print('ROS2 publishers started')
print('  Fixed Frame : apartment/apartment_root')
print('  TF topic    : /tf')
print('  Marker topic: /semworld/viz_marker  (QoS Durability=Transient Local)')

## 3 · Initialise Unified Pipeline

`ActionPipeline.from_world()` auto-detects the robot manipulator and builds
one pipeline object that handles **any** registered action type.

In [ ]:
pipeline = ActionPipeline.from_world(world)

print("ActionPipeline ready")
print(f"  manipulator : {type(pipeline.world_context.manipulator).__name__}")

# Show which action types are registered in the dispatcher
from llmr.pipeline import ActionDispatcher
print(f"  registered action types: {list(ActionDispatcher._registry.keys())}")

In [ ]:
PICKUP_INSTRUCTION = "Pick up the milk from the counter"
PLACE_INSTRUCTION = "Place the milk on the table"

## 4 · Slot Filling — Per-Action Schema

A **single LLM call** (`nodes/slot_filler.py`) classifies the instruction into
an action type *and* extracts all relevant parameters.  The result is a clean,
typed schema specific to that action:

- `PickUpSlotSchema` — fields: `object_description`, `arm`, `grasp_params`
- `PlaceSlotSchema`  — fields: `object_description`, `arm`, `target_description`

Both `object_description` and `target_description` are `EntityDescriptionSchema`
instances (defined once in `schemas/common.py`) — a target surface is described
with the same four fields (`name`, `semantic_type`, `spatial_context`, `attributes`)
as any pickable object.  No mixed optional fields from other action types leak through.

In [ ]:
pickup_schema: PickUpSlotSchema = pipeline.classify_and_extract(PICKUP_INSTRUCTION)

print(f"Instruction : {PICKUP_INSTRUCTION!r}")
print(f"Schema type : {type(pickup_schema).__name__}  ← typed by LLM classification")
print()
print("PickUpSlotSchema:")
print(f"  action_type          : {pickup_schema.action_type}")
print(f"  object.name          : {pickup_schema.object_description.name}")
print(f"  object.semantic_type : {pickup_schema.object_description.semantic_type}")
print(f"  object.spatial       : {pickup_schema.object_description.spatial_context}")
print(f"  arm                  : {pickup_schema.arm}   ← null = unspecified")
print(f"  grasp_params         : {pickup_schema.grasp_params}   ← null = unspecified")

In [ ]:
place_schema: PlaceSlotSchema = pipeline.classify_and_extract(PLACE_INSTRUCTION)

print(f"Instruction : {PLACE_INSTRUCTION!r}")
print(f"Schema type : {type(place_schema).__name__}  ← typed by LLM classification")
print()
print("PlaceSlotSchema:")
print(f"  action_type               : {place_schema.action_type}")
print(f"  object.name               : {place_schema.object_description.name}")
print(f"  object.semantic_type      : {place_schema.object_description.semantic_type}")
print(f"  arm                       : {place_schema.arm}   ← null = unspecified")
print()
# target_description is EntityDescriptionSchema — same four fields as object_description
print(f"  target.name               : {place_schema.target_description.name}")
print(f"  target.semantic_type      : {place_schema.target_description.semantic_type}")
print(f"  target.spatial_context    : {place_schema.target_description.spatial_context}")
print(f"  target.attributes         : {place_schema.target_description.attributes}")
print()
print(f"  type(object_description)  : {type(place_schema.object_description).__name__}")
print(f"  type(target_description)  : {type(place_schema.target_description).__name__}  ← same EntityDescriptionSchema")

## 5 · PickUp — Full Pipeline Run

`pipeline.run()` handles the entire chain:
1. **Phase 1** — `nodes/slot_filler.py`: one LLM call → `PickUpSlotSchema`
2. **Grounding** — `entity_grounder.py`: resolves `"milk"` to the `milk.stl` world body
3. **Phase 2** — `nodes/resolver.py` via `run_pickup_resolver()`: LLM reasons over world
   context to fill `arm`, `approach_direction`, `vertical_alignment`, `rotate_gripper`
4. Returns a fully specified `PickUpAction`

In [ ]:
pickup_action = pipeline.run(PICKUP_INSTRUCTION)

gd = pickup_action.grasp_description
obj_name = str(getattr(getattr(pickup_action.object_designator, "name", None), "name",
                        pickup_action.object_designator))

print(f"PickUpAction resolved from: {PICKUP_INSTRUCTION!r}")
print()
print(f"  object_designator  : {obj_name}")
print(f"  arm                : {pickup_action.arm}")
print(f"  approach_direction : {gd.approach_direction}")
print(f"  vertical_alignment : {gd.vertical_alignment}")
print(f"  rotate_gripper     : {gd.rotate_gripper}")
print(f"  manipulator        : {type(gd.manipulator).__name__}")

In [ ]:
# type(pickup_action)

In [ ]:
# from pycram.robot_plans import (
#     ParkArmsActionDescription,
#     MoveTorsoActionDescription,
#     NavigateActionDescription,
#     PickUpActionDescription,
#     PlaceActionDescription,
# )
# from pycram.datastructures.enums import ApproachDirection, VerticalAlignment, Arms
#
# gd = GraspDescription(approach_direction=pickup_action.grasp_description.approach_direction, vertical_alignment=VerticalAlignment.NoAlignment, rotate_gripper=pickup_action.grasp_description.rotate_gripper, manipulator=pickup_action.grasp_description.manipulator)
#
# ad = PickUpActionDescription(object_designator=pickup_action.object_designator,arm=pickup_action.arm,grasp_description=gd)

In [ ]:
# ad.to_dict()

In [ ]:
# with simulated_robot:
#     SequentialPlan(context, ad).perform()

## 6 · Place — Full Pipeline Run

The **same** `pipeline.run()` call now handles a place instruction:
1. **Phase 1** — `nodes/slot_filler.py`: one LLM call → `PlaceSlotSchema`
2. **Grounding** — `entity_grounder.py`: grounds **two** `EntityDescriptionSchema`
   instances — the object and the target surface (same schema type, different roles)
3. **Phase 2** — `nodes/resolver.py` via `run_place_resolver()`: LLM resolves arm
   based on target surface position relative to robot
4. Returns a fully specified `PlaceAction`

In [ ]:
place_action = pipeline.run(PLACE_INSTRUCTION)

obj_name = str(getattr(getattr(place_action.object_designator, "name", None), "name",
                        place_action.object_designator))
tgt_name = str(getattr(getattr(place_action.target_location, "name", None), "name",
                        place_action.target_location))

print(f"PlaceAction resolved from: {PLACE_INSTRUCTION!r}")
print()
print(f"  object_designator  : {obj_name}")
print(f"  arm                : {place_action.arm}")
print(f"  target_location    : {tgt_name}")

In [ ]:
# from pycram.robot_plans import (
#     ParkArmsActionDescription,
#     MoveTorsoActionDescription,
#     NavigateActionDescription,
#     PickUpActionDescription,
#     PlaceActionDescription,
# )
# from pycram.datastructures.enums import ApproachDirection, VerticalAlignment, Arms
#
# ad = PlaceActionDescription(object_designator=place_action.object_designator,target_location=place_action.target_location, arm=Arms.RIGHT)

In [ ]:
# with simulated_robot:
#     SequentialPlan(context, ad).perform()

## 7 · Step-by-Step: Classify → Dispatch

`pipeline.run()` is a convenience wrapper around two explicit steps:

```python
schema = pipeline.classify_and_extract(instruction)  # LLM call → PickUpSlotSchema | PlaceSlotSchema
action = pipeline.dispatch(schema)                   # ground + resolve → concrete action
```

Splitting the steps is useful for inspection or overriding schema fields
before committing to execution (e.g. forcing a specific arm).

In [ ]:
# ── Step 1: classify + extract ─────────────────────────────────────────────────
schema = pipeline.classify_and_extract(PICKUP_INSTRUCTION)
print(f"classify_and_extract({PICKUP_INSTRUCTION!r})")
print(f"  → action_type = {schema.action_type}")
print(f"  → object      = {schema.object_description.name!r}")
print()

# (optional) inspect / override before dispatching
# e.g. schema.arm = "LEFT"  # force left arm

# ── Step 2: dispatch to handler ───────────────────────────────────────────────
action = pipeline.dispatch(schema)
gd = action.grasp_description
print(f"dispatch(schema) → {type(action).__name__}")
print(f"  arm                : {action.arm}")
print(f"  approach_direction : {gd.approach_direction}")
print(f"  vertical_alignment : {gd.vertical_alignment}")
print(f"  rotate_gripper     : {gd.rotate_gripper}")

In [ ]:
# ── Step 1: classify + extract ─────────────────────────────────────────────────
schema = pipeline.classify_and_extract(PLACE_INSTRUCTION)
print(f"classify_and_extract({PLACE_INSTRUCTION!r})")
print(f"  → action_type  = {schema.action_type}")
print(f"  → object       = {schema.object_description.name!r}")
print(f"  → target       = {schema.target_description.name!r}")
print()

# ── Step 2: dispatch to handler ───────────────────────────────────────────────
action = pipeline.dispatch(schema)
obj_name = str(getattr(getattr(action.object_designator, "name", None), "name",
                        action.object_designator))
tgt_name = str(getattr(getattr(action.target_location, "name", None), "name",
                        action.target_location))
print(f"dispatch(schema) → {type(action).__name__}")
print(f"  object_designator  : {obj_name}")
print(f"  arm                : {action.arm}")
print(f"  target_location    : {tgt_name}")

## 8 · Both Actions Back-to-Back

To make the contrast clear: the same three lines of code work for any instruction,
regardless of whether it's a pick-up, a place, or any other registered action type.

In [ ]:
instructions = [
    "Pick up the milk from the counter",
    "Place the milk on the table_area_main",
]
plans = []
for instr in instructions:
    action = pipeline.run(instr)
    plans.append(action)
    print(f"Instruction : {instr!r}")
    print(f"  → {type(action).__name__}")
    print(f"     arm = {action.arm}")
    if hasattr(action, "grasp_description") and action.grasp_description:
        gd = action.grasp_description
        print(f"     approach_direction = {gd.approach_direction}")
        print(f"     vertical_alignment = {gd.vertical_alignment}")
        print(f"     rotate_gripper     = {gd.rotate_gripper}")
    if hasattr(action, "target_location") and action.target_location is not None:
        tgt = str(getattr(getattr(action.target_location, "name", None), "name",
                          action.target_location))
        print(f"     target_location    = {tgt}")
    print()

In [ ]:
plans[-1]

In [ ]:
from pycram.robot_plans import (
    ParkArmsActionDescription,
    MoveTorsoActionDescription,
    NavigateActionDescription,
    PickUpActionDescription,
    PlaceActionDescription,
)
from pycram.datastructures.enums import ApproachDirection, VerticalAlignment, Arms
from pycram.datastructures.pose import PoseStamped

gd = GraspDescription(approach_direction=plans[0].grasp_description.approach_direction, vertical_alignment=VerticalAlignment.NoAlignment, rotate_gripper=plans[0].grasp_description.rotate_gripper, manipulator=plans[0].grasp_description.manipulator)

pickad = PickUpActionDescription(object_designator=plans[0].object_designator,arm=plans[0].arm,grasp_description=gd)
placead = PlaceActionDescription(object_designator=plans[1].object_designator,target_location=PoseStamped.from_matrix(plans[-1].target_location.global_pose, frame=world.root), arm=Arms.RIGHT)

In [ ]:
PoseStamped.from_matrix(plans[-1].target_location.global_pose, frame=world.root)

In [ ]:
type(plans[-1].target_location.global_pose)

In [ ]:
with simulated_robot:
    SequentialPlan(context, pickad, placead).perform()

## 9 · Generic Resolver — Calling Phase 2 Directly

`nodes/resolver.py` exposes a single `run_resolver()` function that works for
**any** action type.  The typed wrappers (`run_pickup_resolver`, `run_place_resolver`)
are thin convenience calls on top of it.

```python
# Under the hood, run_pickup_resolver() is just:
run_resolver(
    world_context=...,
    known_parameters=...,
    parameters_to_resolve=...,
    prompt=pick_up_resolver_prompt,      # injected
    schema_cls=PickUpDiscreteResolutionSchema,  # injected
)
```

Adding a **new action type** (e.g. `PushAction`) requires only:
1. A new `PushDiscreteResolutionSchema` in `schemas/`
2. A new prompt in `prompts/`
3. A three-line typed wrapper in `nodes/resolver.py`

No changes to `resolver.py`'s graph logic, `action_pipeline.py`, or `action_dispatcher.py`.

In [ ]:
from llmr.workflows.prompts.pick_up import pick_up_resolver_prompt
from llmr.workflows.schemas.pick_up import PickUpDiscreteResolutionSchema

# Demonstrate run_resolver() called directly with an explicit world context string.
# In normal use the handler builds this string from live world poses;
# here we construct a minimal example manually.
_world_ctx = (
    "Robot position: x=0.000, y=0.000, z=0.000\n"
    "Object 'milk': x=1.200, y=-0.350, z=0.850\n"
    "  → 1.20m in front of and 0.35m to the right of the robot, "
    "0.85m above robot origin.\n"
    "  → Semantic types: Milk, FoodItem\n"
    "  → Bounding box (w×d×h): 0.070 × 0.070 × 0.200 m"
)

resolution = run_resolver(
    world_context=_world_ctx,
    known_parameters="None – all discrete parameters are unspecified.",
    parameters_to_resolve=(
        "arm  (choose LEFT or RIGHT based on object position)\n"
        "approach_direction  (FRONT / BACK / LEFT / RIGHT)\n"
        "vertical_alignment  (TOP / BOTTOM / NoAlignment)\n"
        "rotate_gripper      (true / false)"
    ),
    prompt=pick_up_resolver_prompt,
    schema_cls=PickUpDiscreteResolutionSchema,
)

print("run_resolver() output (PickUpDiscreteResolutionSchema):")
print(f"  arm                : {resolution.arm}")
print(f"  approach_direction : {resolution.approach_direction}")
print(f"  vertical_alignment : {resolution.vertical_alignment}")
print(f"  rotate_gripper     : {resolution.rotate_gripper}")
print(f"  reasoning          : {resolution.reasoning}")